# LLM Zoomcamp 2026 — Módulo 5: Monitoring (Homework)

**Autor:** PTeixeira.Tech
**Curso:** [LLM Zoomcamp](https://github.com/DataTalksClub/llm-zoomcamp/) · [enunciado](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/05-monitoring/homework.md)

Neste homework troco o stack manual do módulo (dataclass + PostgreSQL + Streamlit/Grafana) por **OpenTelemetry (OTel)** — o padrão de instrumentação sobre o qual Logfire, Langfuse e Phoenix são construídos. Uso o mesmo RAG de course-lessons do HW1 (72 lesson pages indexadas com minsearch), instrumento `rag`/`search`/`llm` com traces, gravo os spans em SQLite e analiso os dados de trace.

> **Nota de método:** deixei o notebook *runnable* de ponta a ponta. As respostas das múltiplas escolhas eu **não** cravo aqui — cada questão termina com uma célula que imprime o valor observado e um campo `Minha resposta:` pra eu marcar depois de rodar contra a minha própria key. Q2 e Q3 variam entre execuções; Q1, Q4, Q5 e Q6 são determinísticas pela estrutura.


## Setup

Ambiente e download dos arquivos (executo isto no terminal, **antes** de abrir o notebook — deixo `rag_helper.py` e `starter.py` na mesma pasta):

```bash
mkdir llm-zoomcamp-hw5 && cd llm-zoomcamp-hw5
uv init
uv add gitsource minsearch openai python-dotenv
uv add opentelemetry-api opentelemetry-sdk

PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring
wget $PREFIX/rag_helper.py
wget $PREFIX/starter.py
```

E a key no `.env`:

```
OPENAI_API_KEY=sk-...
```

Com os arquivos na pasta, o notebook apenas **importa** as classes e funções de que precisa (`RAGBase` de `rag_helper`, e `rag`/`index`/`client` de `starter`). A célula abaixo só confere que os dois arquivos estão presentes.

In [1]:
import pathlib

for fname in ("rag_helper.py", "starter.py"):
    assert pathlib.Path(fname).exists(), (
        f"{fname} não encontrado — baixe com wget (ver célula de Setup) antes de continuar"
    )
print("rag_helper.py e starter.py presentes na pasta")


rag_helper.py e starter.py presentes na pasta


In [3]:
# Carrega a key do .env
import os
from dotenv import load_dotenv
load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Defina OPENAI_API_KEY no .env antes de continuar"
print("OPENAI_API_KEY carregada")


OPENAI_API_KEY carregada


## OpenTelemetry — tracer provider

Configuro o `TracerProvider` **antes** de importar/usar `starter`, para o tracer estar pronto antes de qualquer código que crie spans. Começo com o `ConsoleSpanExporter` (síncrono, imediato — bom pra desenvolvimento) para inspecionar visualmente o que o OTel captura.

In [4]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")
print("tracer pronto:", tracer)


tracer pronto: <opentelemetry.sdk.trace.Tracer object at 0x11233c440>


In [5]:
# O starter carrega os 72 lessons, monta o índice minsearch e expõe `rag` (RAGBase),
# além de `index` e `client`. Importado DEPOIS do tracer provider estar configurado.
from starter import rag, index, client
from rag_helper import RAGBase

QUERY = "How does the agentic loop keep calling the model until it stops?"
print("RAG base pronto:", rag)


RAG base pronto: <rag_helper.RAGBase object at 0x13019aa50>


## Função de custo

O enunciado pede para computar o custo a partir de input/output tokens "usando o código dos módulos anteriores". Deixo os preços como constantes **ajustáveis** — confirme os valores vigentes do `gpt-5.4-mini` na tabela da OpenAI; os números abaixo são placeholders.

> Atenção (custo real em produção): `gpt-5.4-mini` é um modelo de *reasoning*. Se você fatura só input+output, subestima — os *reasoning tokens* contam como output. Para custo fiel, some `usage.output_tokens_details.reasoning_tokens`. Para o homework não muda a resposta marcada.

In [8]:
# USD por 1M tokens — AJUSTE conforme a tabela vigente da OpenAI
PRICE_INPUT_PER_1M  = 0.25   # placeholder
PRICE_OUTPUT_PER_1M = 2.00   # placeholder

def calc_cost(input_tokens: int, output_tokens: int) -> float:
    return input_tokens / 1e6 * PRICE_INPUT_PER_1M + output_tokens / 1e6 * PRICE_OUTPUT_PER_1M


## Q1 — First trace

Crio um `RAGTraced(RAGBase)` que envolve `rag()`, `search()` e `llm()`, cada um no seu próprio span. Como `RAGBase.rag()` chama `self.search()` e `self.llm()`, e `self` é uma instância de `RAGTraced`, os spans de `search` e `llm` ficam **aninhados** dentro do span `rag`.

Já incluo aqui os atributos de tokens/custo no span `llm` (isso serve para a Q2 — evita reescrever a classe).

**Pergunta:** rodando a query, quantos spans o trace produz? (`1` / `3` / `5` / `7`)

In [9]:
class RAGTraced(RAGBase):

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            span.set_attribute("query", query)
            results = super().search(query, num_results=num_results)
            span.set_attribute("num_results", len(results))
            return results

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            usage = response.usage
            # Q2: métricas como atributos do span
            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)
            span.set_attribute("cost", calc_cost(usage.input_tokens, usage.output_tokens))
            span.set_attribute("model", self.model)
            return response

    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            span.set_attribute("query", query)
            return super().rag(query)


rag_traced = RAGTraced(index=index, llm_client=client)


In [10]:
# Roda a query — o ConsoleSpanExporter imprime cada span finalizado como um dict (ReadableSpan).
# Conte quantos blocos ReadableSpan aparecem na saída abaixo.
answer = rag_traced.rag(QUERY)
print("\n===== RESPOSTA DO RAG =====\n")
print(answer)


{
    "name": "search",
    "context": {
        "trace_id": "0xfb926d986e83af8ad988887a60008266",
        "span_id": "0x696e1bc019b16f2b",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x08c1a1d39acbd20e",
    "start_time": "2026-07-20T18:12:37.050406Z",
    "end_time": "2026-07-20T18:12:37.052486Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "f16f99c3-4b8a-431d-af00-c18879ee08dc",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xfb926d986e83af8ad

**Q1 — leia a saída acima e conte os blocos `ReadableSpan`.**

Estrutura esperada: `rag` (pai) → `search` + `llm`.



## Q2 — Capturing metrics as span attributes

O span `llm` já grava `input_tokens`, `output_tokens` e `cost` (via `set_attribute`). Re-rodo a query e leio o `input_tokens`.

**Pergunta:** quantos input tokens vemos? (`700` / `7000` / `70000` / `700000`) — os números variam entre runs, escolha o mais próximo.

In [11]:
# Captura o input_tokens diretamente do response para inspeção (o mesmo valor vai pro span)
resp = rag_traced.llm(rag_traced.build_prompt(QUERY, rag_traced.search(QUERY)))
usage = resp.usage
print("input_tokens :", usage.input_tokens)
print("output_tokens:", usage.output_tokens)
print("cost (USD)   :", calc_cost(usage.input_tokens, usage.output_tokens))


{
    "name": "search",
    "context": {
        "trace_id": "0x9f1de22afdea70825f5f2904894209a0",
        "span_id": "0xb79ce51320e9a080",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-20T18:19:12.286328Z",
    "end_time": "2026-07-20T18:19:12.290398Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "f16f99c3-4b8a-431d-af00-c18879ee08dc",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xb4da8cc083f5d9d2002e244d867ea38d"

## Q3 — Span timing

Cada span registra a própria duração. Comparo a duração do span `search` com a do span `llm`. Nesta célula eu meço os tempos de parede em Python para ter os números lado a lado (o valor do span é equivalente).

**Pergunta:** para uma query típica, quanto tempo leva a chamada do LLM? (`<100ms` / `100–500ms` / `500–2000ms` / `>2000ms`) — a primeira chamada pode ter cold start; pegue a faixa mais frequente.

In [12]:
import time

t0 = time.perf_counter(); results = rag_traced.search(QUERY); t1 = time.perf_counter()
prompt = rag_traced.build_prompt(QUERY, results)
t2 = time.perf_counter(); _ = rag_traced.llm(prompt); t3 = time.perf_counter()

print(f"search: {(t1-t0)*1000:8.1f} ms")
print(f"llm   : {(t3-t2)*1000:8.1f} ms")


{
    "name": "search",
    "context": {
        "trace_id": "0x272eb82cd0f981fb1f32516b60ba1475",
        "span_id": "0x9ef4b6d299e12dbf",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-20T18:19:53.346121Z",
    "end_time": "2026-07-20T18:19:53.348035Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "f16f99c3-4b8a-431d-af00-c18879ee08dc",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x651918866293bd5e78529a3b3dc9db51"

## Q4 — Saving traces to SQLite

Troco o destino: em vez de imprimir, persisto cada span num SQLite. A instrumentação não muda — só crio um `SpanExporter` customizado e ligo via `SimpleSpanProcessor`.

> **Nota de kernel:** o OTel só aceita **um** `TracerProvider` global por processo (chamar `set_tracer_provider` de novo é ignorado). Então eu *adiciono* o processor do SQLite ao provider já existente — a partir daqui os spans vão para console **e** para o banco.

**Pergunta:** quais nomes de span aparecem na tabela `spans`? (`só rag` / `rag e llm` / `rag, search e llm` / `search, llm e judge`)

In [13]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult

DB_PATH = "traces.db"

class SQLiteSpanExporter(SpanExporter):
    def __init__(self, db_path=DB_PATH):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self, timeout_millis=30000):
        return True


In [14]:
# Começo o banco do zero para a análise ficar limpa
import os
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

provider.add_span_processor(SimpleSpanProcessor(SQLiteSpanExporter(DB_PATH)))
print("SQLite exporter ligado ->", DB_PATH)


SQLite exporter ligado -> traces.db


In [15]:
# Re-roda a query (agora grava no SQLite). É a 1a das 4 execuções que a Q6 vai precisar.
_ = rag_traced.rag(QUERY)

import sqlite3
con = sqlite3.connect(DB_PATH)
names = con.execute("SELECT DISTINCT name FROM spans ORDER BY name").fetchall()
print("span names na tabela:", [n[0] for n in names])
con.close()


{
    "name": "search",
    "context": {
        "trace_id": "0x375e0ca8eb8a7ed1a5db9ac2ab663c19",
        "span_id": "0xcd04354fca954b53",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xf31bf6dd1f4a72f6",
    "start_time": "2026-07-20T18:22:16.453071Z",
    "end_time": "2026-07-20T18:22:16.454777Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "f16f99c3-4b8a-431d-af00-c18879ee08dc",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x375e0ca8eb8a7ed1a

## Q5 — Querying trace data

O span `rag` engloba tudo (search + llm), então eu o **excluo** e comparo os filhos. Somo a duração por nome de span (`end_time - start_time`, em nanossegundos).

**Pergunta:** qual tipo de span consome mais tempo total? (`search` / `llm` / são parecidos)

In [16]:
import pandas as pd, sqlite3

con = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("SELECT * FROM spans", con)
con.close()

df["duration_ms"] = (df["end_time"] - df["start_time"]) / 1e6  # ns -> ms
total = (
    df[df["name"] != "rag"]
    .groupby("name")["duration_ms"]
    .sum()
    .sort_values(ascending=False)
)
print(total)


name
llm       5889.250
search       1.706
Name: duration_ms, dtype: float64


## Q6 — Token stability across runs

Rodo a **mesma** query mais 3 vezes (4 chamadas `rag` no total) e comparo `input_tokens` de cada span `llm`. Como o minsearch é determinístico (TF-IDF, sem randomness), a expectativa é que o contexto recuperado — e portanto os input tokens — seja estável.

**Pergunta:** quanto variam os input tokens nas 4 runs? (`idênticos` / `dentro de 10%` / `dentro de 50%` / `mais de 50%`)

In [17]:
# Faltam 3 execuções para totalizar 4
for _ in range(3):
    _ = rag_traced.rag(QUERY)

con = sqlite3.connect(DB_PATH)
llm_tokens = pd.read_sql_query(
    "SELECT input_tokens FROM spans WHERE name = 'llm' ORDER BY start_time", con
)["input_tokens"]
con.close()

print("input_tokens por run:", llm_tokens.tolist())
if len(llm_tokens) and llm_tokens.min():
    spread = (llm_tokens.max() - llm_tokens.min()) / llm_tokens.min() * 100
    print(f"variacao (max-min)/min: {spread:.2f}%")
    print("identicos?" , bool(llm_tokens.nunique() == 1))


{
    "name": "search",
    "context": {
        "trace_id": "0xe7d59c4b5acce786a7a67ad27a0a5316",
        "span_id": "0x3ef0176ecffbff19",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x924b70fdf18df4b3",
    "start_time": "2026-07-20T18:22:54.593781Z",
    "end_time": "2026-07-20T18:22:54.595404Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "f16f99c3-4b8a-431d-af00-c18879ee08dc",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xe7d59c4b5acce786a